# RimGraph-DG V4.5 — Kaggle GPU ORIGA-first run
Use this when Colab GPU quota is exhausted. In Kaggle: add the **Glaucoma Datasets** input (`arnavjain1/glaucoma-datasets`), turn **Internet ON**, and select a **GPU accelerator**. This launcher runs the decoded-mask audit first, skips baseline retraining to conserve GPU quota, and trains a fresh RimGraph model only for held-out ORIGA.

In [ ]:
from pathlib import Path
import hashlib, json, traceback, urllib.request
import torch

DATA_DIR = Path('/kaggle/input/glaucoma-datasets')
if not DATA_DIR.exists():
    raise RuntimeError("Dataset not attached. Kaggle: Add Input -> search 'Glaucoma Datasets' by arnavjain1 -> Add, then rerun.")

print('=== V4.5 KAGGLE GPU CHECK ===', flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not active. Notebook Settings -> Accelerator -> GPU, restart session, then rerun.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('Dataset:', DATA_DIR, flush=True)
print('===============================', flush=True)

GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': str(DATA_DIR),
    'fast_dev_run': False,
    'run_name': 'paper_run_v45_kaggle_origa',
    'code_revision': 'rimgraph-dg-v4.5-20260809-kaggle',
    'seeds': [2029],
    'fold_targets': ['ORIGA'],
    'run_global_baseline': False,
    'run_full_model': True,
    'baseline_reuse_run': '',
    'run_optuna': False,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'full_epochs': 30,
    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,
    'patience': 8,
    'mixed_precision': True,
    'pretrained': True,
    'backbone': 'convnext_tiny.fb_in22k_ft_in1k',
    'backbone_fallback': 'convnext_tiny',
    'num_workers': 0,
    'n_visual_examples': 2,
    'resume': True,
}

COMMIT = 'f33460ecdf26b527e1740ae607025e0bb2ddcd3a'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]

print('[LAUNCHER] downloading pinned runner source ...', flush=True)
try:
    raw_code = '\n'.join(urllib.request.urlopen(f'{ROOT}/{name}', timeout=60).read().decode('utf-8') for name in parts)
except Exception as exc:
    raise RuntimeError('Could not download pinned GitHub runner. In Kaggle Settings turn Internet ON, then rerun.') from exc
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'Raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
    ('runner_patch_v44_runtime.py', 'apply_v44_runtime'),
    ('runner_patch_v45_masks.py', 'apply_v45_masks'),
    ('runner_patch_v45_lowlabels.py', 'apply_v45_lowlabels'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    print(f'[LAUNCHER] applying {patch_name}', flush=True)
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}', timeout=60).read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v45_kaggle_origa.py', 'exec')
print('[LAUNCHER] V4.5 Kaggle assembly PASSED', flush=True)

try:
    exec(code, globals(), globals())
    local_run = Path.cwd() / 'Glaucomma_runs' / 'paper_run_v45_kaggle_origa'
    completion = local_run / 'RUN_COMPLETED.json'
    if not completion.exists():
        raise RuntimeError(f'Runner returned without completion marker: {completion}')
    zip_path = Path(str(local_run) + '.zip')
    print('\n✅ V4.5 KAGGLE VERIFIED COMPLETION', flush=True)
    print('Results:', local_run, flush=True)
    print('ZIP:', zip_path, flush=True)
except BaseException:
    trace = traceback.format_exc()
    print('\n=== V4.5 KAGGLE FAILURE TRACEBACK ===', flush=True)
    print(trace, flush=True)
    failure = Path.cwd() / 'RimGraph_V45_KAGGLE_FAILURE_TRACEBACK.txt'
    try:
        failure.write_text(trace, encoding='utf-8')
        print('Failure trace saved:', failure, flush=True)
    except Exception:
        pass
    raise
